In [1]:
import sys
import numpy as np
import pybullet as p
import time
import logging
from typing import List, Tuple, Optional, Sequence, Collection, Dict, Any, cast
import random
import json

from predicators.structs import Action, Array, GroundAtom, Object, State, Type, ParameterizedOption
from predicators import utils
from predicators.settings import CFG
from gym.spaces import Box

#Import core environment methods, robot function etc.

from predicators.envs.pybullet_blocks import PyBulletBlocksEnv
from predicators.envs.pybullet_env import PyBulletEnv, create_pybullet_block
from predicators.pybullet_helpers.robots import SingleArmPyBulletRobot
from predicators.pybullet_helpers.robots.mobile_single_arm import MobileSingleArmPyBulletRobot
from predicators.pybullet_helpers.geometry import Pose
from predicators.pybullet_helpers.joint import JointPositions, get_joint_infos, get_joint_positions
from predicators.pybullet_helpers.link import get_link_state

#Import the functions that are to be tested:

from predicators.pybullet_helpers.motion_planning import run_motion_planning, run_base_motion_planning,\
                                                            run_coordinated_motion_planning
#The pick/place options to be tested are accessed via the env instance
from predicators.pybullet_helpers.controllers import execute_coordinated_path, create_move_end_effector_to_pose_option,\
                                                    create_change_fingers_option, create_move_base_option

#Configure logging for better debugging outputs:
#logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logging.basicConfig(
    level=logging.WARNING,                    
    format="%(asctime)s %(name)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

#Defining test configuration, and overriding some default ones:
CFG.pybullet_robot = "fetch_mobile"
CFG.use_gui = True
#Draws helpful debug lines in the workspace.
#NOT SURE WHETHER TO USE THIS. WILL DECIDE AFTER A COUPLE RUNS.
#CFG.pybullet_draw_debug = True
#Initializing with standard size of blocks.
CFG.blocks_block_size = 0.05
CFG.pybullet_birrt_num_iters = 50
CFG.pybullet_birrt_num_attempts = 10
CFG.pybullet_birrt_smooth_amt = 20
CFG.seed = random.randint(0,10000)
#CFG.seed = 12
#Num of PyBullet physics steps per high-level Action in visualize_action_sequence
CFG.pybullet_sim_steps_per_action = 10

pybullet build time: Jan 29 2025 23:16:28


In [2]:
#Helper functions:
#Function to reset robot to a known pose
def reset_robot_fetch_mobile(robot: MobileSingleArmPyBulletRobot,
                             physics_client_id:int,
                             base_pose: Tuple[float, float, float] = (1.35, 0.75, 0.0), # (x,y,theta)
                             arm_joint_angle: Optional[List[float]]=None):
    """
    Resets the robot's base/ teleports it and arm to specified poses.

    """

    robot.move_base_to(base_pose, physics_client_id)
    if arm_joint_angle:
        #Set arm joints only
        robot.set_joints(arm_joint_angle)
    else:
        #robot.initial_joint_positions includes arm and finger joints
        robot.set_joints(robot.initial_joint_positions)
    #Step simulation a bit to allow PyBullet to settle the state.
    for _ in range(10):
        p.stepSimulation(physicsClientId=physics_client_id)


#Fn to create blocks in the env.
def create_test_block(env: PyBulletEnv,
                      pose: Tuple[float, float, float],
                      color: Tuple[float, float, float, float] = (0.8, 0.2, 0.2, 1.0),
                      name_suffix: str = "test") -> int:
    """
    Creates a single block at a specified pose for testing and returns its PyBullet ID.
    """

    #Use the fn defined in utils to create block
    block_id = create_pybullet_block(
        color,
        (CFG.blocks_block_size/2,)*3,
        env._obj_mass,
        env._obj_friction,
        env._default_orn,
        env._physics_client_id
    )
    #Place the block at desired pose.
    p.resetBasePositionAndOrientation(block_id, pose, env._default_orn, physicsClientId=env._physics_client_id)

    return block_id


#Get the list of all bodies except the robot.
#TODO: Need to add logic that saves object/body name
#      which can be used for better debugging with collision.
def get_all_non_robot_bodies(robot_id: int, physics_client_id:int) -> List[int]:
    """
    Gets all PyBullet body IDs in the simulation except for the robot itself.
    These are typically used as collision obstacles.
    """
    all_bodies = [p.getBodyUniqueId(i, physicsClientId=physics_client_id)
                    for i in range(p.getNumBodies(physicsClientId=physics_client_id))]


    return [b for b in all_bodies if b!=robot_id]

In [3]:
#Setup Env.
#Initialize the PyBulletBlocksEnv which sets up PyBullet,
#loads the robot, tables etc.

# use_gui flag is set to false here to ensure that this can run even without a OpenGL/ GUI setup;
# to set see rendering, set use_gui=True or use_gui=CFG.use_gui

env = PyBulletBlocksEnv(use_gui=False)

# Resets the environment to a specific task, getting an initial symbolic state.
# While initial_state_from_env is fetched, the option tests will create their own
# more specific symbolic states.
initial_state = env.reset("train", 0)

# The robot instance from the environment
robot = env._pybullet_robot
# The PyBullet physics client ID
physics_client_id = env._physics_client_id

assert isinstance(robot, MobileSingleArmPyBulletRobot), f"\n{robot} should be an instance of MobileSingleArmPyBulletRobot."

# dyn = p.getDynamicsInfo(robot.robot_id, -1, physicsClientId=env._physics_client_id)
# print(f"\nMass, inertialFrame…{dyn}")
# input()

logging.info(f"Using robot: {robot.get_name()}")


#Store the robot's default arm and finger joint positions.
home_arm_joints = robot.initial_joint_positions

robot_obj = initial_state.get_objects(env._robot_type)[0]

#Store permament, fixed bodies
static_collision_bodies = get_all_non_robot_bodies(robot.robot_id, physics_client_id)

#print(f"Static_collision_bodies:{static_collision_bodies}")

#sys.exit(0)

#Define a rectangular workspace for base motion planning tests.
#(min_x, min_y, max_x, max_y)
workspace_bounds = (1.0, 0.2, 1.7, 1.3)

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
head_camera_linkb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
head_camera_rgb_frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
head_camera_rgb_optical_frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Impo

/home/cloaked04/anaconda3/envs/predicators/lib/python3.10/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


In [4]:
mark_pos = (1.5, 0.75, CFG.blocks_block_size / 2 + env.table_height + 0.1)

p.addUserDebugText(
    "*",                          
    mark_pos,                     
    textColorRGB=[0, 0, 0],       
    textSize=1,                 
    lifeTime=0,                   
    physicsClientId=physics_client_id)

-1

In [7]:
initial_base_pose_5 = (0.4, 0.6, -np.pi/2)
reset_robot_fetch_mobile(robot, physics_client_id, base_pose=initial_base_pose_5, arm_joint_angle=home_arm_joints)

# Create a block to be picked.
block_to_pick_pose_world = (1.5, 0.75, CFG.blocks_block_size / 2 + env.table_height)
logging.critical(f"World coords of block to pick:{block_to_pick_pose_world}.")

block_to_pick_id = create_test_block(env, pose=block_to_pick_pose_world, name_suffix="pick_target")

# Make a unique name for the symbolic object
symbolic_block_name = f"block{block_to_pick_id}" 
block_to_pick_obj_sym = Object(symbolic_block_name, env._block_type)

# Step 1: Update the environment's physical-to-symbolic map.
# This tells _get_state() that the new physical block ID now corresponds
# to a new symbolic object.
env._block_id_to_block[block_to_pick_id] = block_to_pick_obj_sym

#logging.debug(f"\nBlocks in the env:{env._block_id_to_block}.")

# Step 2: Get a handle to the official state object, which is mutable.
state_obj_to_modify = env._current_observation
#logging.debug(f"\nCurrent environment: {state_obj_to_modify}.")
assert isinstance(state_obj_to_modify, utils.PyBulletState), \
    f"Expected env._current_observation to be a PyBulletState, got {type(state_obj_to_modify)}"

state_obj_to_modify.data[block_to_pick_obj_sym] = np.zeros(len(env._block_type.feature_names))
fresh_state = env._get_state()
state_obj_to_modify.data = fresh_state.data
state_obj_to_modify.simulator_state = fresh_state.simulator_state 
state_obj_to_modify.base_pose = fresh_state.base_pose

2025-07-24 13:07:20 root [CRITICAL] World coords of block to pick:(1.5, 0.75, 0.225).


In [8]:
# Crucial: ensure robot starts not holding anything. PyBulletEnv uses this.
env._held_obj_id = None
# The symbolic robot object, already in env.types and env._get_state()
robot_obj_sym = env._robot

#Simple check to affirm that the block is part of the sim/env.
bodies_in_sim = get_all_non_robot_bodies(robot.robot_id, physics_client_id)
assert block_to_pick_id in bodies_in_sim, f"\n Block to be picked not part of sim."

In [18]:
for item in env._current_state.data:
    print(f"\n{item}: {env._current_state.data[item]}")
    print(f"\n---------------------------------------")


robby:robot: [ 0.4        -0.00359999  0.54079163  1.        ]

---------------------------------------

block0:block: [1.3713416  0.48926318 0.22498469 0.         0.05419374 0.01303867
 0.82653326]

---------------------------------------

block1:block: [1.3713241  0.48926705 0.27495956 0.         0.0669353  0.2800048
 0.38854155]

---------------------------------------

block2:block: [1.3712889  0.48927844 0.32494792 0.         0.6255852  0.16042829
 0.07346   ]

---------------------------------------

block3:block: [1.3726995  0.6992064  0.22498956 0.         0.11181556 0.4910257
 0.29365885]

---------------------------------------

block11:block: [1.5   0.75  0.225 0.    0.8   0.2   0.2  ]

---------------------------------------


In [22]:
env._current_state.simulator_state

[-0.594813934798528,
 0.08213050150912982,
 1.720118807675905,
 1.5765014729785563,
 1.4885996110790467,
 -1.7200825589224584,
 0.9693904345953313,
 0.04,
 0.04]

In [24]:
home_arm_joints

[-0.594813934798528,
 0.08213050150912982,
 1.720118807675905,
 1.5765014729785563,
 1.4885996110790467,
 -1.7200825589224584,
 0.9693904345953313,
 0.04,
 0.04]

In [33]:
initial_state

PyBulletState(data={block0:block: array([1.3713471 , 0.4892623 , 0.225     , 0.        , 0.05419374,
       0.01303867, 0.82653326], dtype=float32), block1:block: array([1.3713471 , 0.4892623 , 0.275     , 0.        , 0.0669353 ,
       0.2800048 , 0.38854155], dtype=float32), block2:block: array([1.3713471 , 0.4892623 , 0.325     , 0.        , 0.6255852 ,
       0.16042829, 0.07346   ], dtype=float32), block3:block: array([1.3727001 , 0.6992063 , 0.225     , 0.        , 0.11181556,
       0.4910257 , 0.29365885], dtype=float32), robby:robot: array([1.       , 0.3      , 0.5000001, 1.       ], dtype=float32)}, simulator_state=[-0.594813934798528, 0.11218094288507019, 1.7566148992319965, 1.5911679755197872, 1.4604720740400103, -1.7577149294578562, 0.975539052941321, 0.03999999910593033, 0.03999999910593033], base_pose=(0.4, 0.3, 0.0))

In [27]:
env._get_state()

PyBulletState(data={robby:robot: array([ 0.4       , -0.00359999,  0.54079163,  1.        ], dtype=float32), block0:block: array([1.3713416 , 0.48926318, 0.22498469, 0.        , 0.05419374,
       0.01303867, 0.82653326], dtype=float32), block1:block: array([1.3713241 , 0.48926705, 0.27495956, 0.        , 0.0669353 ,
       0.2800048 , 0.38854155], dtype=float32), block2:block: array([1.3712889 , 0.48927844, 0.32494792, 0.        , 0.6255852 ,
       0.16042829, 0.07346   ], dtype=float32), block3:block: array([1.3726995 , 0.6992064 , 0.22498956, 0.        , 0.11181556,
       0.4910257 , 0.29365885], dtype=float32), block11:block: array([1.5  , 0.75 , 0.225, 0.   , 0.8  , 0.2  , 0.2  ], dtype=float32)}, simulator_state=[-0.594813934798528, 0.08213050150912982, 1.720118807675905, 1.5765014729785563, 1.4885996110790467, -1.7200825589224584, 0.9693904345953313, 0.04, 0.04], base_pose=(0.4, 0.6, -1.5707963267948963))

In [32]:
env._current_state

PyBulletState(data={robby:robot: array([ 0.4       , -0.00359999,  0.54079163,  1.        ], dtype=float32), block0:block: array([1.3713416 , 0.48926318, 0.22498469, 0.        , 0.05419374,
       0.01303867, 0.82653326], dtype=float32), block1:block: array([1.3713241 , 0.48926705, 0.27495956, 0.        , 0.0669353 ,
       0.2800048 , 0.38854155], dtype=float32), block2:block: array([1.3712889 , 0.48927844, 0.32494792, 0.        , 0.6255852 ,
       0.16042829, 0.07346   ], dtype=float32), block3:block: array([1.3726995 , 0.6992064 , 0.22498956, 0.        , 0.11181556,
       0.4910257 , 0.29365885], dtype=float32), block11:block: array([1.5  , 0.75 , 0.225, 0.   , 0.8  , 0.2  , 0.2  ], dtype=float32)}, simulator_state=[-0.594813934798528, 0.08213050150912982, 1.720118807675905, 1.5765014729785563, 1.4885996110790467, -1.7200825589224584, 0.9693904345953313, 0.04, 0.04], base_pose=(0.4, 0.6, -1.5707963267948963))

In [43]:
env._current_observation

PyBulletState(data={robby:robot: array([ 0.4       , -0.00359999,  0.54079163,  1.        ], dtype=float32), block0:block: array([1.3713416 , 0.48926318, 0.22498469, 0.        , 0.05419374,
       0.01303867, 0.82653326], dtype=float32), block1:block: array([1.3713241 , 0.48926705, 0.27495956, 0.        , 0.0669353 ,
       0.2800048 , 0.38854155], dtype=float32), block2:block: array([1.3712889 , 0.48927844, 0.32494792, 0.        , 0.6255852 ,
       0.16042829, 0.07346   ], dtype=float32), block3:block: array([1.3726995 , 0.6992064 , 0.22498956, 0.        , 0.11181556,
       0.4910257 , 0.29365885], dtype=float32), block11:block: array([1.5  , 0.75 , 0.225, 0.   , 0.8  , 0.2  , 0.2  ], dtype=float32)}, simulator_state=[-0.594813934798528, 0.08213050150912982, 1.720118807675905, 1.5765014729785563, 1.4885996110790467, -1.7200825589224584, 0.9693904345953313, 0.04, 0.04], base_pose=(0.4, 0.6, -1.5707963267948963))

In [36]:
initial_state.get_objects(env._robot_type)

robby:robot

In [39]:
env.action_space

Box([-1.6056 -1.221     -inf -2.251     -inf -2.16      -inf  0.      0.    ], [1.6056 1.518     inf 2.251     inf 2.16      inf 0.05   0.05  ], (9,), float32)

In [42]:
held_obj = env._detect_held_object()
print(held_obj)

None


In [45]:
env._extract_robot_state(env._current_state)

array([ 0.4       , -0.00359999,  0.54079163,  0.7071    ,  0.        ,
       -0.7071    ,  0.        ,  0.04      ], dtype=float32)

In [47]:
env.get_name()

'pybullet_blocks'

In [54]:
env._get_object_ids_for_held_check()

[3, 4, 5, 6, 11]

________________________________________________________________________________________________________________________________________________________

Trying out functions in blocks.py:

In [116]:
env._block_type

Type(name='block')

In [117]:
env._robot_type

Type(name='robot')

In [133]:
print(f"{type(env._On)}")
print(env._On.name)
print(env._On.types)
print(env._On.arity)

<class 'predicators.structs.Predicate'>
On
[Type(name='block'), Type(name='block')]
2


In [135]:
env._On.pretty_str()

('?x:block, ?y:block', 'On(?x, ?y)')

In [136]:
env._On.pddl_str()

'(On ?x0 - block ?x1 - block)'

In [137]:
env._On.get_negation()

NOT-On

In [138]:
type(env._On.get_negation())

predicators.structs.Predicate

In [141]:
print(f"The robot initialized in non PyBullet environments is an Object with attributes name, type:")
print(f"\n Robot name: {env._robot.name}")
print(f"\n Robot type: {env._robot.type}")

The robot initialized in non PyBullet environments is an Object with attributes name, type:

 Robot name: robby

 Robot type: Type(name='robot')


The two functions below use env._get_tasks() defined in blocks.py to get train and test tasks:

In [142]:
env._generate_train_tasks()

[EnvironmentTask(init_obs=PyBulletState(data={block0:block: array([1.32841512, 0.68679279, 0.225     , 0.        , 0.58371183,
        0.97916985, 0.24911052]), block1:block: array([1.32841512, 0.68679279, 0.275     , 0.        , 0.4622052 ,
        0.64458689, 0.9597161 ]), block2:block: array([1.32841512, 0.68679279, 0.325     , 0.        , 0.78445209,
        0.75008345, 0.78309474]), block3:block: array([1.32841512, 0.68679279, 0.375     , 0.        , 0.83009726,
        0.70448488, 0.03491255]), robby:robot: array([1. , 0.3, 0.5, 1. ], dtype=float32)}, simulator_state=[-0.594813934798528, 0.11218094288507019, 1.7566148992319965, 1.5911679755197872, 1.4604720740400103, -1.7577149294578562, 0.975539052941321, 0.03999999910593033, 0.03999999910593033], base_pose=None), goal_description={On(block3:block, block2:block), OnTable(block0:block), OnTable(block2:block), On(block1:block, block0:block)}),
 EnvironmentTask(init_obs=PyBulletState(data={block0:block: array([1.36912235, 1.0774928

In [143]:
env._generate_test_tasks()

[EnvironmentTask(init_obs=PyBulletState(data={block0:block: array([1.36826555, 0.63818369, 0.225     , 0.        , 0.40200212,
        0.92920356, 0.77522922]), block1:block: array([1.36826555, 0.63818369, 0.275     , 0.        , 0.39373497,
        0.22380438, 0.89184165]), block2:block: array([1.36826555, 0.63818369, 0.325     , 0.        , 0.35628039,
        0.53739426, 0.90355536]), block3:block: array([1.36826555, 0.63818369, 0.375     , 0.        , 0.47641115,
        0.11861917, 0.08096364]), block4:block: array([1.36826555, 0.63818369, 0.425     , 0.        , 0.11097007,
        0.52319448, 0.79273021]), block5:block: array([1.36826555, 0.63818369, 0.475     , 0.        , 0.19396608,
        0.25096614, 0.27533784]), robby:robot: array([1. , 0.3, 0.5, 1. ], dtype=float32)}, simulator_state=[-0.594813934798528, 0.11218094288507019, 1.7566148992319965, 1.5911679755197872, 1.4604720740400103, -1.7577149294578562, 0.975539052941321, 0.03999999910593033, 0.03999999910593033], base_

In [146]:
env.predicates

{Clear, GripperOpen, Holding, On, OnTable}

In [147]:
env.goal_predicates

{On, OnTable}

In [149]:
env.types

{Type(name='block'), Type(name='robot')}

______________________________________________________________________________________________________________________________________________________

Inspecting Robot:
_______________________________________________________________________________________________________________________________

In [57]:
robot.get_name()

'fetch_mobile'

In [62]:
robot.action_space

Box([-1.6056 -1.221     -inf -2.251     -inf -2.16      -inf  0.      0.    ], [1.6056 1.518     inf 2.251     inf 2.16      inf 0.05   0.05  ], (9,), float32)

In [63]:
robot.end_effector_name

'gripper_axis'

In [64]:
robot.end_effector_id

17

In [65]:
robot.tool_link_name

'gripper_link'

In [66]:
robot.tool_link_id

17

In [67]:
robot.wrist_roll_link_name

'wrist_roll_link'

In [68]:
robot.wrist_roll_link_id

16

In [69]:
robot.base_link_name

'base_link'

In [70]:
robot.arm_joints

[10, 11, 12, 13, 14, 15, 16, 19, 18]

In [71]:
robot.arm_joint_names

['shoulder_pan_joint',
 'shoulder_lift_joint',
 'upperarm_roll_joint',
 'elbow_flex_joint',
 'forearm_roll_joint',
 'wrist_flex_joint',
 'wrist_roll_joint',
 'l_gripper_finger_joint',
 'r_gripper_finger_joint']

In [73]:
for info in robot.joint_infos:
    print(f"\n{info}")
    print(f"\n----------------------------------------")


JointInfo(jointIndex=0, jointName='r_wheel_joint', jointType=0, qIndex=7, uIndex=6, flags=1, jointDamping=0.0, jointFriction=0.0, jointLowerLimit=0.0, jointUpperLimit=-1.0, jointMaxForce=8.85, jointMaxVelocity=17.4, linkName='r_wheel_link', jointAxis=(0.0, 1.0, 0.0), parentFramePos=(0.0048914, -0.18738, 0.053925), parentFrameOrn=(3.0615e-17, 0.0, 0.0, 1.0), parentIndex=-1)

----------------------------------------

JointInfo(jointIndex=1, jointName='l_wheel_joint', jointType=0, qIndex=8, uIndex=7, flags=1, jointDamping=0.0, jointFriction=0.0, jointLowerLimit=0.0, jointUpperLimit=-1.0, jointMaxForce=8.85, jointMaxVelocity=17.4, linkName='l_wheel_link', jointAxis=(0.0, 1.0, 0.0), parentFramePos=(0.0048914, 0.18738, 0.053925), parentFrameOrn=(3.0615e-17, 0.0, 0.0, 1.0), parentIndex=-1)

----------------------------------------

JointInfo(jointIndex=2, jointName='torso_lift_joint', jointType=4, qIndex=-1, uIndex=-1, flags=0, jointDamping=100.0, jointFriction=0.0, jointLowerLimit=0.0, join

In [75]:
robot.joint_names

['r_wheel_joint',
 'l_wheel_joint',
 'torso_lift_joint',
 'head_pan_joint',
 'head_tilt_joint',
 'head_camera_joint',
 'head_camera_rgb_joint',
 'head_camera_rgb_optical_joint',
 'head_camera_depth_joint',
 'head_camera_depth_optical_joint',
 'shoulder_pan_joint',
 'shoulder_lift_joint',
 'upperarm_roll_joint',
 'elbow_flex_joint',
 'forearm_roll_joint',
 'wrist_flex_joint',
 'wrist_roll_joint',
 'gripper_axis',
 'r_gripper_finger_joint',
 'l_gripper_finger_joint',
 'bellows_joint2',
 'estop_joint',
 'laser_joint',
 'torso_fixed_joint']

In [76]:
for joint_name in robot.joint_names:
    print(f"\n Joint info for {joint_name}: {robot.joint_info_from_name(joint_name)}")
    print(f"\n----------------------------------------------------------------------")


 Joint info for r_wheel_joint: JointInfo(jointIndex=0, jointName='r_wheel_joint', jointType=0, qIndex=7, uIndex=6, flags=1, jointDamping=0.0, jointFriction=0.0, jointLowerLimit=0.0, jointUpperLimit=-1.0, jointMaxForce=8.85, jointMaxVelocity=17.4, linkName='r_wheel_link', jointAxis=(0.0, 1.0, 0.0), parentFramePos=(0.0048914, -0.18738, 0.053925), parentFrameOrn=(3.0615e-17, 0.0, 0.0, 1.0), parentIndex=-1)

----------------------------------------------------------------------

 Joint info for l_wheel_joint: JointInfo(jointIndex=1, jointName='l_wheel_joint', jointType=0, qIndex=8, uIndex=7, flags=1, jointDamping=0.0, jointFriction=0.0, jointLowerLimit=0.0, jointUpperLimit=-1.0, jointMaxForce=8.85, jointMaxVelocity=17.4, linkName='l_wheel_link', jointAxis=(0.0, 1.0, 0.0), parentFramePos=(0.0048914, 0.18738, 0.053925), parentFrameOrn=(3.0615e-17, 0.0, 0.0, 1.0), parentIndex=-1)

----------------------------------------------------------------------

 Joint info for torso_lift_joint: JointI

In [79]:
robot.link_from_name(info.linkName)

23

In [80]:
robot.left_finger_joint_name

'l_gripper_finger_joint'

In [81]:
robot.right_finger_joint_name

'r_gripper_finger_joint'

In [82]:
robot.left_finger_id

19

In [83]:
robot.right_finger_id

18

In [84]:
robot.left_finger_joint_idx

7

In [86]:
robot.right_finger_joint_idx

8

In [92]:
print(f"\nRobot arm joint lower limits:{robot.joint_lower_limits}")
print(f"\n-------------------------------------------------------")
print(f"\nRobot arm joint upper limits:{robot.joint_upper_limits}")
print(f"\n-------------------------------------------------------")
print(f"\nEnvironment's Action Space lower limit: {env.action_space.low}")
print(f"\n-------------------------------------------------------")
print(f"\nEnvironment's Action Space upper limit: {env.action_space.high}")


Robot arm joint lower limits:[-1.6056, -1.221, -inf, -2.251, -inf, -2.16, -inf, 0.0, 0.0]

-------------------------------------------------------

Robot arm joint upper limits:[1.6056, 1.518, inf, 2.251, inf, 2.16, inf, 0.05, 0.05]

-------------------------------------------------------

Environment's Action Space lower limit: [-1.6056 -1.221     -inf -2.251     -inf -2.16      -inf  0.      0.    ]

-------------------------------------------------------

Environment's Action Space upper limit: [1.6056 1.518     inf 2.251     inf 2.16      inf 0.05   0.05  ]


In [93]:
robot.open_fingers

0.04

In [96]:
robot.closed_fingers

0.01

In [99]:
#Used at the start of this notebook to get robot's initial joint positions as well:
robot.initial_joint_positions

[-0.594813934798528,
 0.08213050150912982,
 1.720118807675905,
 1.5765014729785563,
 1.4885996110790467,
 -1.7200825589224584,
 0.9693904345953313,
 0.04,
 0.04]

In [103]:
robot.get_joints()

[-0.594813934798528,
 0.08213050150912982,
 1.720118807675905,
 1.5765014729785563,
 1.4885996110790467,
 -1.7200825589224584,
 0.9693904345953313,
 0.04,
 0.04]

In [101]:
#These correspond to: rx, ry, rz, qx, qy, qz, qw, rf
#Positions: rx, ry, rz
#Quaternions: qx, qy, qz, qw
#Finger: rf
robot.get_state()

array([ 0.4       , -0.00359999,  0.54079163,  0.5       , -0.5       ,
       -0.5       , -0.5       ,  0.04      ], dtype=float32)

In [105]:
#Just setting the finger params from open to closed:
robot.set_joints([-0.594813934798528,
 0.08213050150912982,
 1.720118807675905,
 1.5765014729785563,
 1.4885996110790467,
 -1.7200825589224584,
 0.9693904345953313,
 0.01,
 0.01])

In [106]:
#Fingers should now be closed:
robot.get_joints()

[-0.594813934798528,
 0.08213050150912982,
 1.720118807675905,
 1.5765014729785563,
 1.4885996110790467,
 -1.7200825589224584,
 0.9693904345953313,
 0.01,
 0.01]

In [107]:
#From mobile_single_arm.py
robot.wheel_ids

[1, 0]

In [108]:
robot._wheel_radius

0.065

In [109]:
robot._wheel_separation

0.3748

In [110]:
robot.footprint_radius

0.2174

In [115]:
robot.wheel_joint_names

('l_wheel_joint', 'r_wheel_joint')